# Geometry-V1 Batch 2B
PREPARED_NOT_EXECUTED. User-run Colab only; no current push, Colab, Drive, or model-execution authorization.
The fixed RGB input below is operational identity only; science_denominator=0.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
import hashlib,json,os,pathlib,shutil,subprocess,sys,zipfile
import numpy as np
from PIL import Image
from google.colab import userdata
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Geometry-V1'
EXECUTION_EXACT='2044187863797a619fd0fd56311e766dafbf75d5'; RUN_ID='geometry-v1-b2b-204418786379-operational-01'
PROPOSED_PENDING_FINAL_USER_CONFIRMATION="/content/drive/MyDrive/CEG-WM/Geometry-V1/Batch2B"
DRIVE_ROOT=pathlib.Path(PROPOSED_PENDING_FINAL_USER_CONFIRMATION); repo=pathlib.Path('/content/geometry-v1-source'); run_dir=DRIVE_ROOT/RUN_ID
root_key=''; hf_token=''; runner_env=None; process=None; input_paths=[]; fixed_image=None; fixed_array=None
input_dir=pathlib.Path('/content/geometry-v1-inputs'); work=pathlib.Path('/content/geometry-v1-receipt'); archive=pathlib.Path('/content')/(RUN_ID+'.zip'); sidecar=pathlib.Path('/content')/(RUN_ID+'.zip.sha256')
def build_fixed_operational_rgb() -> Image.Image:
 yy, xx = np.indices((512, 512), dtype=np.uint16)
 red = (3*xx + 5*yy) % 256
 green = (7*xx + 2*yy + 17*(xx//32)) % 256
 blue = (xx ^ (3*yy)) % 256
 array = np.stack((red, green, blue), axis=-1).astype(np.uint8)
 array[40:180,55:225] = (241,67,31)
 array[300:465,335:493] = (19,181,223)
 return Image.fromarray(array, mode='RGB')
def verify_checkout():
 h=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip(); c=subprocess.run(['git','status','--porcelain'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
 if h!=EXECUTION_EXACT or c: raise RuntimeError('checkout identity differs')
def exclusive_copy(source,target):
 with source.open('rb') as read, target.open('xb') as write:
  while chunk:=read.read(1048576): write.write(chunk)
def parse_child(rc,stdout):
 if len(stdout)>4096: raise RuntimeError('bounded stdout exceeded')
 lines=stdout.decode('utf-8','strict').splitlines(); success='CEGWM_GEOMETRY_V1_OPERATIONAL_PREFLIGHT '; failure='CEGWM_GEOMETRY_V1_OPERATIONAL_FAILURE '
 if len(lines)!=1 or not lines[0]: raise RuntimeError('exactly one receipt line required')
 if lines[0].startswith(success): prefix,status=success,'success'
 elif lines[0].startswith(failure): prefix,status=failure,'failure'
 else: raise RuntimeError('invalid receipt prefix')
 payload=json.loads(lines[0][len(prefix):])
 if not isinstance(payload,dict) or (rc==0)!=(status=='success'): raise RuntimeError('receipt/return-code mismatch')
 return status,payload,lines[0]
try:
 if repo.exists() or run_dir.exists() or input_dir.exists() or work.exists() or archive.exists() or sidecar.exists(): raise FileExistsError('create-only path exists')
 subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
 subprocess.run(['git','checkout','--detach',EXECUTION_EXACT],cwd=repo,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); verify_checkout()
 subprocess.run([sys.executable,'-m','pip','install',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); verify_checkout()
 input_dir.mkdir()
 fixed_image=build_fixed_operational_rgb(); fixed_array=np.asarray(fixed_image)
 if fixed_image.mode!='RGB' or fixed_image.size!=(512,512) or fixed_array.dtype!=np.uint8 or fixed_array.shape!=(512,512,3): raise RuntimeError('fixed RGB input identity differs')
 fixed_path=input_dir/'geometry_v1_batch2b_fixed_rgb.png'
 with fixed_path.open('xb') as handle: fixed_image.save(handle,format='PNG')
 input_paths.append(fixed_path)
 hf_token=userdata.get('HF_TOKEN'); root_key=userdata.get('CEG_WM_ROOT_KEY')
 if not hf_token or not root_key: raise RuntimeError('required secret unavailable')
 runner_env={n:v for n,v in os.environ.items() if all(x not in n.upper() for x in ('TOKEN','KEY','SECRET'))}; runner_env['HF_TOKEN']=hf_token; runner_env['CEG_WM_ROOT_KEY']=root_key
 command=[sys.executable,'-m','experiments.run_geometry_v1_qk_operational_preflight','--repo-root',str(repo),'--expected-exact',EXECUTION_EXACT,str(fixed_path)]
 process=subprocess.Popen(command,cwd=repo,env=runner_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL); stdout,_=process.communicate(timeout=1800); status,payload,line=parse_child(process.returncode,stdout)
 work.mkdir(); status_name='success.json' if status=='success' else 'failure.json'
 for name,value in [('receipt.json',payload),(status_name,{'status':payload.get('status')})]:
  with (work/name).open('x',encoding='utf-8') as handle: json.dump(value,handle,sort_keys=True,separators=(',',':'))
 allowed=['receipt.json',status_name,'manifest.json','SHA256SUMS']
 with (work/'manifest.json').open('x',encoding='utf-8') as handle: json.dump({'execution_exact':EXECUTION_EXACT,'run_id':RUN_ID,'allowed_filenames':allowed},handle,sort_keys=True,separators=(',',':'))
 with (work/'SHA256SUMS').open('x',encoding='ascii') as handle:
  for name in allowed[:-1]: handle.write(hashlib.sha256((work/name).read_bytes()).hexdigest()+'  '+name+'\n')
 with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED) as bundle:
  for name in allowed: bundle.write(work/name,name)
 with archive.open('rb') as handle: digest=hashlib.sha256(handle.read()).hexdigest()
 with sidecar.open('xb') as handle: handle.write((digest+'  '+archive.name+'\n').encode('ascii'))
 DRIVE_ROOT.mkdir(parents=True,exist_ok=True); run_dir.mkdir(); exclusive_copy(archive,run_dir/archive.name); exclusive_copy(sidecar,run_dir/sidecar.name)
 print(line)
 if status=='failure': raise RuntimeError('child reported sanitized failure')
finally:
 root_key=''; hf_token=''; fixed_image=None; fixed_array=None
 if runner_env is not None:
  runner_env.pop('HF_TOKEN',None); runner_env.pop('CEG_WM_ROOT_KEY',None)
 if process is not None and process.poll() is None: process.kill(); process.wait()
 for path in input_paths:
  if path.exists(): path.unlink()
 if input_dir.exists(): input_dir.rmdir()
 for path in (work,archive,sidecar):
  if path.is_dir(): shutil.rmtree(path)
  elif path.exists(): path.unlink()
